# Predictive Modeling: Environmental Changes and Impacts

This notebook develops one-year-ahead country-level models using `global_climate_impact_1990_2025.csv`. It predicts two environmental outcomes: annual mean PM2.5 concentration and temperature anomaly. The models use measurements from year *t* to predict the outcome for year *t + 1*.

## Modeling workflow
1. Create adjacent country-year feature/target pairs and exclude pairs with missing target measurements.
2. Fit ridge regression models with training data through 2016.
3. Select regularization using 2017–2019 validation years.
4. Evaluate once on a later 2020–2025 holdout, compared with a persistence baseline (next year equals the previous year).

Preprocessing statistics are learned from the fitting period only. These are exploratory historical models, not operational forecasts, causal estimates, or regulatory measurements. Country-level data do not represent local exposure.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

CSV_NAME = "global_climate_impact_1990_2025.csv"
data_path = next(
    (parent / CSV_NAME for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / CSV_NAME).is_file()),
    None,
)
if data_path is None:
    raise FileNotFoundError(
        f"Could not find {CSV_NAME!r} in the current directory or its parents."
    )

df = pd.read_csv(data_path)
print(f"Loaded {len(df):,} rows for {df['Country'].nunique():,} countries")
print(f"Year range: {df['Year'].min()}–{df['Year'].max()}")

Loaded 6,948 rows for 193 countries
Year range: 1990–2025


## Define target outcomes and available-at-prediction features

For each outcome, the feature row represents the current year and its label is the same country's next-year measurement. Country and continent are context features; all numeric indicators are from the feature year, avoiding use of target-year measurements.

In [2]:
models_to_run = {
    "PM2.5 (µg/m³)": {
        "target": "PM2_5_Annual_Mean_ug_m3",
        "features": [
            "Year", "PM2_5_Annual_Mean_ug_m3", "PM10_Annual_Mean_ug_m3",
            "Air_Quality_Index", "Ozone_Annual_Mean_ug_m3",
            "NO2_Annual_Mean_ug_m3", "CO2_Emissions_Total_Mt",
            "Population_Year_Millions", "GDP_Current_USD_Billions",
            "Renewable_Energy_Share_Pct",
        ],
        "unit": "µg/m³",
    },
    "Temperature anomaly (°C)": {
        "target": "Temperature_Anomaly_C",
        "features": [
            "Year", "Temperature_Anomaly_C", "Annual_Mean_Temp_C",
            "CO2_Emissions_Total_Mt", "Methane_Emissions_MtCO2e",
            "PM2_5_Annual_Mean_ug_m3", "Renewable_Energy_Share_Pct",
            "Forest_Area_Pct_Of_Land", "Drought_Index_0to100",
        ],
        "unit": "°C",
    },
}
categorical_features = ["Country", "Continent"]
required = set(categorical_features)
for specification in models_to_run.values():
    required.add(specification["target"])
    required.update(specification["features"])
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"CSV is missing modeling columns: {sorted(missing)}")

print("Model targets:", ", ".join(models_to_run))
print("Training targets: through 2016 | validation: 2017–2019 | holdout: 2020–2025")

Model targets: PM2.5 (µg/m³), Temperature anomaly (°C)
Training targets: through 2016 | validation: 2017–2019 | holdout: 2020–2025


## Preprocessing and model helpers

Missing numeric feature values are imputed with training-period medians, with missingness indicators retained. Numeric variables are standardized using training-only statistics. Categorical values are one-hot encoded, and unseen validation/test categories receive all-zero indicators.

In [3]:
def make_next_year_pairs(data, target):
    """Build adjacent-year country pairs with an observed next-year target."""
    ordered = data.sort_values(["Country", "Year"]).copy()
    groups = ordered.groupby("Country", sort=False)
    ordered["Target_Year"] = groups["Year"].shift(-1)
    ordered["Target_Value"] = groups[target].shift(-1)
    pairs = ordered.loc[
        ordered["Target_Year"].eq(ordered["Year"] + 1)
        & ordered[target].notna()
        & ordered["Target_Value"].notna()
    ].copy()
    pairs["Target_Year"] = pairs["Target_Year"].astype(int)
    return pairs


def prepare_features(fit_rows, other_rows, feature_columns):
    """Fit imputation, scaling, and encoding on fit_rows only."""
    fit_numeric = fit_rows[feature_columns].astype(float).copy()
    other_numeric = other_rows[feature_columns].astype(float).copy()
    fit_missing = fit_numeric.isna().astype(float)
    other_missing = other_numeric.isna().astype(float)
    fit_missing.columns = [f"{name}_was_missing" for name in feature_columns]
    other_missing.columns = fit_missing.columns

    medians = fit_numeric.median().fillna(0.0)
    fit_numeric = fit_numeric.fillna(medians)
    other_numeric = other_numeric.fillna(medians)
    means = fit_numeric.mean()
    scales = fit_numeric.std(ddof=0).replace(0, 1.0).fillna(1.0)
    fit_numeric = (fit_numeric - means) / scales
    other_numeric = (other_numeric - means) / scales

    fit_categories = fit_rows[categorical_features].fillna("Unknown").astype(str)
    other_categories = other_rows[categorical_features].fillna("Unknown").astype(str)
    fit_encoded = pd.get_dummies(fit_categories, dtype=float)
    other_encoded = pd.get_dummies(other_categories, dtype=float).reindex(
        columns=fit_encoded.columns, fill_value=0.0
    )
    fit_matrix = pd.concat([fit_numeric, fit_missing, fit_encoded], axis=1)
    other_matrix = pd.concat([other_numeric, other_missing, other_encoded], axis=1)
    other_matrix = other_matrix.reindex(columns=fit_matrix.columns, fill_value=0.0)
    return fit_matrix.to_numpy(), other_matrix.to_numpy()


def fit_ridge(features, targets, alpha):
    """Fit ridge regression with an unpenalized intercept."""
    design = np.column_stack([np.ones(len(features)), features])
    penalty = np.eye(design.shape[1]) * alpha
    penalty[0, 0] = 0.0
    return np.linalg.solve(design.T @ design + penalty, design.T @ targets)


def predict_ridge(features, coefficients):
    return np.column_stack([np.ones(len(features)), features]) @ coefficients


def regression_metrics(actual, predicted):
    errors = np.asarray(actual) - np.asarray(predicted)
    total_variation = np.sum((actual - np.mean(actual)) ** 2)
    return {
        "MAE": np.mean(np.abs(errors)),
        "RMSE": np.sqrt(np.mean(errors**2)),
        "R²": 1 - np.sum(errors**2) / total_variation if total_variation else np.nan,
    }


alpha_values = [0.1, 1.0, 10.0, 100.0, 1000.0]
all_scores = []
all_predictions = []

## Train, tune, and evaluate

Each outcome receives an independent time-based split and model. The holdout period is not used to choose the ridge penalty. Compare all model errors with persistence on the same rows.

In [4]:
for outcome_name, specification in models_to_run.items():
    target = specification["target"]
    features = specification["features"]
    unit = specification["unit"]
    pairs = make_next_year_pairs(df, target)
    train = pairs.loc[pairs["Target_Year"] <= 2016].copy()
    validation = pairs.loc[pairs["Target_Year"].between(2017, 2019)].copy()
    test = pairs.loc[pairs["Target_Year"].between(2020, 2025)].copy()
    if min(len(train), len(validation), len(test)) == 0:
        raise ValueError(f"No usable train, validation, or test rows for {outcome_name}.")

    X_train, X_validation = prepare_features(train, validation, features)
    y_train = train["Target_Value"].to_numpy(dtype=float)
    y_validation = validation["Target_Value"].to_numpy(dtype=float)
    tuning = []
    for alpha in alpha_values:
        coefficients = fit_ridge(X_train, y_train, alpha)
        prediction = predict_ridge(X_validation, coefficients)
        tuning.append({"alpha": alpha, **regression_metrics(y_validation, prediction)})
    tuning_table = pd.DataFrame(tuning).sort_values("MAE")
    best_alpha = float(tuning_table.iloc[0]["alpha"])

    development = pairs.loc[pairs["Target_Year"] <= 2019].copy()
    X_development, X_test = prepare_features(development, test, features)
    y_development = development["Target_Value"].to_numpy(dtype=float)
    y_test = test["Target_Value"].to_numpy(dtype=float)
    final_coefficients = fit_ridge(X_development, y_development, best_alpha)
    model_prediction = predict_ridge(X_test, final_coefficients)
    persistence_prediction = test[target].to_numpy(dtype=float)

    print(f"\n{outcome_name}: {len(pairs):,} adjacent-year pairs; selected alpha={best_alpha:g}")
    display(tuning_table.round(4))
    score_rows = [
        {"outcome": outcome_name, "model": "Ridge regression", **regression_metrics(y_test, model_prediction)},
        {"outcome": outcome_name, "model": "Persistence baseline", **regression_metrics(y_test, persistence_prediction)},
    ]
    all_scores.extend(score_rows)
    display(pd.DataFrame(score_rows).round(4))

    result = test[["Country", "Year", "Target_Year", "Target_Value"]].copy()
    result["Outcome"] = outcome_name
    result["Ridge_Prediction"] = model_prediction
    result["Persistence_Prediction"] = persistence_prediction
    result["Absolute_Error"] = np.abs(y_test - model_prediction)
    all_predictions.append(result)

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(y_test, model_prediction, alpha=0.3, s=18)
    lower = min(y_test.min(), model_prediction.min())
    upper = max(y_test.max(), model_prediction.max())
    ax.plot([lower, upper], [lower, upper], color="black", linestyle="--")
    ax.set(title=f"{outcome_name}: actual vs. predicted (2020–2025)",
           xlabel=f"Observed ({unit})", ylabel=f"Predicted ({unit})")
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

scores = pd.DataFrame(all_scores)
predictions = pd.concat(all_predictions, ignore_index=True)
print("\nCombined chronological holdout summary")
display(scores.round(4))


PM2.5 (µg/m³): 6,004 adjacent-year pairs; selected alpha=1


,alpha,MAE,RMSE,R²
1,1.0,3.2414,4.0271,0.8918
2,10.0,3.3070,4.1362,0.8858
0,0.1,3.3956,4.2313,0.8805
3,100.0,3.4207,4.2733,0.8781
4,1000.0,3.4971,4.3543,0.8735


,outcome,model,MAE,RMSE,R²
0,PM2.5 (µg/m³),Ridge regression,3.3724,4.2432,0.8806
1,PM2.5 (µg/m³),Persistence baseline,3.3194,4.1607,0.8852


<Figure size 700x600 with 1 Axes>


Temperature anomaly (°C): 6,708 adjacent-year pairs; selected alpha=10


,alpha,MAE,RMSE,R²
2,10.0,0.0798,0.1010,0.5663
1,1.0,0.0800,0.1010,0.5660
0,0.1,0.0800,0.1011,0.5658
3,100.0,0.0803,0.1019,0.5583
4,1000.0,0.0929,0.1163,0.4246


,outcome,model,MAE,RMSE,R²
0,Temperature anomaly (°C),Ridge regression,0.0848,0.1067,0.6871
1,Temperature anomaly (°C),Persistence baseline,0.0873,0.1089,0.6737


<Figure size 700x600 with 1 Axes>


Combined chronological holdout summary


,outcome,model,MAE,RMSE,R²
0,PM2.5 (µg/m³),Ridge regression,3.3724,4.2432,0.8806
1,PM2.5 (µg/m³),Persistence baseline,3.3194,4.1607,0.8852
2,Temperature anomaly (°C),Ridge regression,0.0848,0.1067,0.6871
3,Temperature anomaly (°C),Persistence baseline,0.0873,0.1089,0.6737


## Interpretation and limitations

- A lower MAE/RMSE than persistence indicates improvement on this historical holdout only; if the ridge model does not beat persistence, prefer the simpler baseline until the model is improved.
- Ridge coefficients and feature associations are predictive, not causal.
- The temporal holdout evaluates later years for countries represented in training; it does not establish performance for unseen countries or future years beyond 2025.
- Country-level annual summaries obscure local variation and cannot be used as station-level forecasts or regulatory measurements.
- The PM2.5 and temperature indicators, their units, and the temperature-anomaly baseline should be validated against dataset documentation.
- Missing target measurements are excluded; missing features are imputed from the training period and marked with missingness indicators.

**Next steps:** validate data provenance and definitions, add reliable station-level observations and forecast-time covariates, and monitor performance on newly observed years before operational use.